# Hierarchical MAS for a Toy Travel Agency

This notebook shows a **hierarchical multi-agent architecture** using the same travel-planning task.
Here we introduce a manager agent that decides the workflow and delegates work to specialized workers.

## What Changes Compared with the Flat Version?

The travel request and travel catalog stay the same.
The difference is only the orchestration:

- the manager plans the work,
- the workers answer only the manager,
- the manager combines everything into the final itinerary.

This is the simplest way to teach **top-down control** in a MAS.

In [1]:
from typing import Literal

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# One small model keeps the examples easy to compare.
# These notebooks assume OPENAI_API_KEY is already available.
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

USER_REQUEST = """
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan
""".strip()

DESTINATIONS = {
    "Lisbon": {
        "best_period": "April to June",
        "style": "sunny, walkable, relaxed",
        "notes": "great for food, viewpoints, and compact neighborhoods",
    },
    "Barcelona": {
        "best_period": "April to June",
        "style": "lively, artistic, seaside",
        "notes": "strong mix of architecture, beach walks, and tapas",
    },
    "Prague": {
        "best_period": "April to June",
        "style": "historic, compact, lower-cost",
        "notes": "easy sightseeing with a classic old-town atmosphere",
    },
}

FLIGHTS = [
    {
        "destination": "Lisbon",
        "code": "TP-833",
        "price": 180,
        "type": "direct",
        "duration": "3h 05m",
    },
    {
        "destination": "Lisbon",
        "code": "IB-310",
        "price": 150,
        "type": "1 stop",
        "duration": "5h 10m",
    },
    {
        "destination": "Barcelona",
        "code": "VY-611",
        "price": 140,
        "type": "direct",
        "duration": "1h 50m",
    },
    {
        "destination": "Barcelona",
        "code": "IB-220",
        "price": 125,
        "type": "1 stop",
        "duration": "4h 00m",
    },
    {
        "destination": "Prague",
        "code": "FR-721",
        "price": 110,
        "type": "direct",
        "duration": "1h 55m",
    },
    {
        "destination": "Prague",
        "code": "OS-410",
        "price": 145,
        "type": "1 stop",
        "duration": "3h 45m",
    },
]

HOTELS = [
    {
        "destination": "Lisbon",
        "name": "Baixa Stay",
        "price_per_night": 145,
        "style": "central boutique hotel",
    },
    {
        "destination": "Lisbon",
        "name": "River Rooms",
        "price_per_night": 120,
        "style": "simple hotel near transit",
    },
    {
        "destination": "Barcelona",
        "name": "Born Hotel",
        "price_per_night": 160,
        "style": "central design hotel",
    },
    {
        "destination": "Barcelona",
        "name": "Gracia Inn",
        "price_per_night": 130,
        "style": "quiet hotel in a walkable district",
    },
    {
        "destination": "Prague",
        "name": "Old Town House",
        "price_per_night": 115,
        "style": "historic hotel near main sights",
    },
    {
        "destination": "Prague",
        "name": "City Garden Hotel",
        "price_per_night": 95,
        "style": "budget-friendly hotel with tram access",
    },
]

ACTIVITIES = [
    {
        "destination": "Lisbon",
        "name": "Alfama food walk",
        "tag": "food",
        "price": 35,
    },
    {
        "destination": "Lisbon",
        "name": "Belem and river tram day",
        "tag": "culture",
        "price": 25,
    },
    {
        "destination": "Barcelona",
        "name": "Gothic Quarter tapas evening",
        "tag": "food",
        "price": 40,
    },
    {
        "destination": "Barcelona",
        "name": "Sagrada Familia and modernism route",
        "tag": "culture",
        "price": 32,
    },
    {
        "destination": "Prague",
        "name": "Old Town walking tour",
        "tag": "culture",
        "price": 18,
    },
    {
        "destination": "Prague",
        "name": "Czech dinner and jazz night",
        "tag": "food",
        "price": 30,
    },
]


def flights_for(destination: str) -> list[dict]:
    return [flight for flight in FLIGHTS if flight["destination"] == destination]



def hotels_for(destination: str) -> list[dict]:
    return [hotel for hotel in HOTELS if hotel["destination"] == destination]



def activities_for(destination: str) -> list[dict]:
    return [activity for activity in ACTIVITIES if activity["destination"] == destination]



def catalog_text() -> str:
    lines = []

    for destination, info in DESTINATIONS.items():
        lines.append(f"Destination: {destination}")
        lines.append(f"  Best period: {info['best_period']}")
        lines.append(f"  Style: {info['style']}")
        lines.append(f"  Notes: {info['notes']}")
        lines.append("  Flights:")

        for flight in flights_for(destination):
            lines.append(
                f"    - {flight['code']} | {flight['type']} | EUR {flight['price']} | {flight['duration']}"
            )

        lines.append("  Hotels:")
        for hotel in hotels_for(destination):
            lines.append(
                f"    - {hotel['name']} | EUR {hotel['price_per_night']} per night | {hotel['style']}"
            )

        lines.append("  Activities:")
        for activity in activities_for(destination):
            lines.append(
                f"    - {activity['name']} | {activity['tag']} | EUR {activity['price']}"
            )

        lines.append("")

    return "\n".join(lines).strip()


CATALOG_TEXT = catalog_text()
print(USER_REQUEST)

Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan


## Define the Manager and Workers

The manager first creates a small ordered plan.
Then each worker executes one assignment.

We keep the worker interfaces plain on purpose so a beginner can follow the data flow line by line.

In [2]:
class WorkerTask(BaseModel):
    worker: Literal[
        "destination_worker",
        "booking_worker",
        "activity_worker",
    ]
    instruction: str = Field(description="A clear task written by the manager.")


class ManagerPlan(BaseModel):
    tasks: list[WorkerTask]


planner = model.with_structured_output(ManagerPlan)

WORKERS = {
    "destination_worker": "Choose the best destination and travel period.",
    "booking_worker": "Choose one flight and one hotel that fit the request.",
    "activity_worker": "Choose activities that match the traveler profile.",
}


def run_worker(worker_name: str, instruction: str, manager_notes: str) -> str:
    # Workers never talk to each other directly.
    # Everything passes through the manager in a star topology.
    messages = [
        SystemMessage(
            content=(
                "You are a subordinate worker in a hierarchical travel-agency MAS. "
                "You only answer the manager. Do not invent options outside the catalog.\n\n"
                f"Your fixed role: {WORKERS[worker_name]}"
            )
        ),
        HumanMessage(
            content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Catalog:\n{CATALOG_TEXT}\n\n"
                f"Manager notes so far:\n{manager_notes or 'No notes yet.'}\n\n"
                f"Your assigned task:\n{instruction}"
            )
        ),
    ]

    return model.invoke(messages).content

## Execute the Hierarchy

Read the next cell carefully:

1. the manager plans,
2. each worker runs in sequence,
3. the manager synthesizes the final result.

This is a classic **manager -> worker -> manager** pattern.

In [3]:
plan = planner.invoke(
    [
        SystemMessage(
            content=(
                "You are the manager in a hierarchical travel-agency MAS. "
                "Break the work into a short ordered plan using the available workers."
            )
        ),
        HumanMessage(
            content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Available workers:\n{WORKERS}"
            )
        ),
    ]
)

manager_notes = []

# The manager explicitly delegates one task at a time.
for task in plan.tasks:
    notes_text = "\n".join(manager_notes)
    result = run_worker(task.worker, task.instruction, notes_text)
    manager_notes.append(f"{task.worker}: {result}")

worker_output_text = "\n".join(manager_notes)

final_itinerary = model.invoke(
    [
        SystemMessage(
            content=(
                "You are the same manager. Combine the worker outputs into one final travel plan."
            )
        ),
        HumanMessage(
            content=(
                f"Traveler request:\n{USER_REQUEST}\n\n"
                f"Worker outputs:\n{worker_output_text}"
            )
        ),
    ]
).content

print("=== Manager plan ===")
for index, task in enumerate(plan.tasks, start=1):
    print(f"{index}. {task.worker} -> {task.instruction}")

print("\n=== Worker outputs ===")
for item in manager_notes:
    print("-", item)

print("\n=== Final itinerary ===")
print(final_itinerary)

=== Manager plan ===
1. destination_worker -> Select the best destination and travel period for a 4-day spring trip from Rome, considering easy flights and a mid-range budget.
2. booking_worker -> Book one flight and one centrally located hotel that fit the mid-range budget and easy flight requirements for the chosen destination and travel period.
3. activity_worker -> Plan a simple daily itinerary for 4 days that includes a mix of food and cultural activities suitable for the traveler.

=== Worker outputs ===
- destination_worker: I recommend Prague for the 4-day spring trip from Rome. The best period is April to June, which fits the spring timing. Prague offers easy direct flights (FR-721) at EUR 110 and a short flight time of 1h 55m, making travel simple. The city is historic and compact, ideal for a mix of culture and food with a mid-range budget. Hotels like Old Town House at EUR 115 per night provide central accommodation near main sights. Activities such as the Old Town walking 

## Why This Fits the Hierarchical Architecture

- One agent controls the workflow.
- Worker agents do not coordinate with each other directly.
- The final answer is assembled by the manager.

That makes the topology clearly hierarchical rather than peer-to-peer.